# 03 · Patent melting-point set: availability and target-blind census

[Open in Colab](https://colab.research.google.com/github/Singati2/TopoQSPR/blob/repro/colab-first/notebooks/colab/03_external_replication.ipynb) · code commit `104899afec01c366b9c31d11ba311aadbdd432c0` · branch `repro/colab-first`

Doc 15. Default: checks the figshare record (CC0 licence, file size, md5) **without downloading**.
`APPROVE_DOWNLOAD=True` (owner approval, doc 15 §7) downloads the 638 MB SDF and runs the
target-blind census: molblocks only (every SD data field, including the melting point, is cut off
before parsing), package standardisation, removal of every constitution present in Bradley or
OPERA, and the frozen go/no-go counts. The confirmation itself is deliberately not implemented: it
is written only after GO and after its protocol is frozen, before any value is read.
Census: about 1–2 h on 2 CPUs; needs about 3 GB disk.

**How to run.** Runtime → Run all, in a fresh CPU runtime. Edit only the configuration cell.
Everything runs as `python -I docs/research/scripts/...` in subprocesses, the same commands as on
Linux/HPC (listed at the end). A `FAIL` line stops the notebook; its numbers must not be used.
**Status:** local clean-environment execution tested (see notebooks/colab/README.md); **Colab unverified**


In [ ]:
# ===== CONFIGURATION (edit here only) =====
import os, pathlib
REPO_URL  = os.environ.get("NB_REPO_URL", "https://github.com/Singati2/TopoQSPR.git")
GIT_SHA   = os.environ.get("NB_GIT_SHA", "104899afec01c366b9c31d11ba311aadbdd432c0")   # exact commit of the scripts; checked after checkout
MODE      = os.environ.get("NB_MODE", "metadata")      # metadata | census
WORK      = pathlib.Path(os.environ.get("NB_WORK", "/content"))
USE_DRIVE = os.environ.get("NB_USE_DRIVE", "0") == "1"   # persist cache + outputs to your Google Drive
DRIVE_DIR = "/content/drive/MyDrive/topoqspr_repro"      # used only if USE_DRIVE
N_JOBS    = int(os.environ.get("NB_N_JOBS", os.cpu_count() or 1))   # worker processes (Colab free: 2)
APPROVE_DOWNLOAD = False   # owner approval to download ~640 MB (doc 15 §7)


In [ ]:
# ===== SETUP: storage, exact checkout, pinned install (about 2-4 min) =====
import json, subprocess, sys, time
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path(DRIVE_DIR)
else:
    BASE = WORK / "topoqspr_repro"
BASE.mkdir(parents=True, exist_ok=True)
REPO = WORK / "TopoQSPR"
OUT  = BASE / "out"
os.environ["TOPOQSPR_CACHE"] = str(BASE / "cache")   # pinned public downloads + processed records

def sh(cmd, check=True):
    """Run a shell command, stream its output, stop the notebook on failure."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0 and check:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")
    return p.returncode

if not (REPO / ".git").exists():
    sh(f"git init -q {REPO} && git -C {REPO} remote add origin {REPO_URL}")
if sh(f"git -C {REPO} fetch -q --depth 1 origin {GIT_SHA} && git -C {REPO} checkout -q FETCH_HEAD", check=False):
    sh(f"git -C {REPO} fetch -q origin '+refs/heads/*:refs/remotes/origin/*' && git -C {REPO} checkout -q {GIT_SHA}")
head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == GIT_SHA, f"checked out {head}, expected {GIT_SHA}"
print("checked out", head)
sh(f'{sys.executable} -m pip install -q "rdkit==2026.3.6" "chembl_structure_pipeline==1.2.4" "numpy==2.2.6" "pandas==2.3.3" "scipy==1.15.3" "scikit-learn==1.7.2" "openpyxl==3.1.5" "joblib==1.6.0" "xgboost==3.2.0" "requests>=2.31" "networkx>=3.2"')
sh(f"{sys.executable} -m pip install -q --no-deps -e {REPO}")
os.chdir(REPO)
PY = f"{sys.executable} -I"   # every analysis runs in a fresh interpreter with the pinned packages
sh(f"""{PY} -c "import sys; sys.path.insert(0, 'docs/research/scripts'); import repro_common as RC, json; m = RC.runtime_metadata(); print(json.dumps(m, indent=1)); mm = RC.version_mismatches(); print('VERSION MISMATCHES:', mm or 'none'); sys.exit(1 if mm else 0)" """)


In [ ]:
flag = "--approve-download" if APPROVE_DOWNLOAD else ""
sh(f"{PY} docs/research/scripts/patent_census.py {OUT}/03_patent {flag} --n-jobs {N_JOBS}")

**Linux/HPC equivalent:** `python -I docs/research/scripts/patent_census.py OUT/03_patent [--approve-download]`